# 03 — Chunking
Splits documents into overlapping chunks (500 chars, overlap 100) and saves `chunks.jsonl`.

In [ ]:
import json
from pathlib import Path

DOCS_PATH   = Path('data/processed/documents.jsonl')
CHUNKS_PATH = Path('data/processed/chunks.jsonl')

CHUNK_SIZE  = 500
OVERLAP     = 100

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

docs = load_jsonl(DOCS_PATH)
print(f'Loaded {len(docs):,} documents')

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping character-level chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in docs:
    text   = doc.get('case_text', '')
    parts  = chunk_text(text, CHUNK_SIZE, OVERLAP)
    for i, part in enumerate(parts):
        all_chunks.append({
            'chunk_id'  : f"{doc['doc_id']}_chunk_{i}",
            'doc_id'    : doc['doc_id'],
            'case_id'   : doc['case_id'],
            'chunk_idx' : i,
            'text'      : part,
            'year'      : doc.get('year'),
            'pdf_url'   : doc.get('pdf_url', ''),
            'petitioner': doc.get('petitioner', ''),
            'respondent': doc.get('respondent', ''),
            'judge'     : doc.get('judge', ''),
            'decision_date': doc.get('decision_date', ''),
        })

print(f'Created {len(all_chunks):,} chunks from {len(docs):,} documents')
print('Sample chunk:', json.dumps(all_chunks[0], indent=2))

In [ ]:
with open(CHUNKS_PATH, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f'Saved → {CHUNKS_PATH}')